# Code to predict Fair value of Stocks

In [12]:
from lxml import html
import requests
import json
import argparse
from collections import OrderedDict

import warnings
warnings.filterwarnings('ignore')
'''
def parse(ticker):
    url = "https://stockanalysis.com/stocks/{}/financials/cash-flow-statement".format(ticker)
    response = requests.get(url, verify=False)
    parser = html.fromstring(response.content)
    fcfs = parser.xpath('//table[contains(@data-test,"financials")]//tr[td/span/text()[contains(., "Free Cash Flow")]]')[0].xpath('.//td/text()')[1:]
    last_fcf = float(fcfs[0].replace(',', ''))
    
    url = "https://finance.yahoo.com/quote/{}/analysis?p={}".format(ticker, ticker)
    response = requests.get(url, headers={'User-Agent': 'Mozilla/5.0 (Windows NT 6.1; WOW64; rv:20.0) Gecko/20100101 Firefox/20.0'})
    parser = html.fromstring(response.content)
    ge = parser.xpath('//table//tbody//tr')

    for row in ge:
        label = row.xpath("td/span/text()")[0]
        if 'Next 5 Years' in label:
            try:
                ge = float(row.xpath("td/text()")[0].replace('%', ''))
            except:
                ge = []
            break

    url = "https://stockanalysis.com/stocks/{}/".format(ticker)
    response = requests.get(url, verify=False)
    parser = html.fromstring(response.content)
    shares = parser.xpath('//table[@data-test="overview-info"]//tbody//tr[td/text()[contains(., "Shares Out")]]')
    
    shares = shares[0].xpath('td/text()')[2]
    factor = 1000 if 'B' in shares else 1 
    shares = float(shares.replace('B', '').replace('M', '')) * factor

    url = "https://stockanalysis.com/stocks/{}/financials/".format(ticker)
    response = requests.get(url, verify=False)
    parser = html.fromstring(response.content)
    # eps = parser.xpath('//table[contains(@id,"financial-table")]//tr[td/span/text()[contains(., "EPS (Diluted)")]]')[0].xpath('.//td/span/text()')[1:]
    eps = parser.xpath('//table[contains(@data-test,"financials")]//tr[td/span/text()[contains(., "EPS (Diluted)")]]')[0].xpath('.//td/text()')[1:]
    eps = float(eps[0].replace(",", ""))
    market_price = float(parser.xpath('//div[@class="price-ext"]/text()')[0].replace('$', '').replace(',', ''))
    return {'fcf': last_fcf, 'ge': ge, 'yr': 5, 'dr': 10, 'pr': 2.5, 'shares': shares, 'eps': eps, 'mp': market_price}
'''

# ...existing code...
# ...existing code...
def parse(ticker):
    def first_or(err_msg, lst):
        if not lst:
            raise ValueError(err_msg)
        return lst[0]

    headers = {'User-Agent': 'Mozilla/5.0'}
    url = f"https://stockanalysis.com/stocks/{ticker}/financials/cash-flow-statement"
    r = requests.get(url, headers=headers, verify=False, timeout=15)
    if r.status_code != 200:
        raise ConnectionError(f"Failed to fetch {url} (status {r.status_code})")
    parser = html.fromstring(r.content)

    # gather all rows from the financials table and normalize text
    rows = parser.xpath('//table[contains(@data-test,"financials")]//tr') or []
    def row_cells(row):
        return [t.strip() for t in row.xpath('.//td//text()') if t.strip()]

    def find_numeric(label_keywords):
        for row in rows:
            cells = row_cells(row)
            if not cells:
                continue
            label = cells[0].lower()
            if any(k.lower() in label for k in label_keywords):
                # take the first numeric column after the label (common layout)
                for val in cells[1:]:
                    v = val.replace(',', '').replace('$', '').strip()
                    try:
                        return float(v)
                    except Exception:
                        continue
        return None

    # try free cash flow first
    last_fcf = find_numeric(['free cash flow', 'free cash flow (fcf)'])

    # fallback: compute FCF = operating cash flow - capital expenditures
    if last_fcf is None:
        ocf = find_numeric(['operating cash flow', 'net cash from operating activities', 'net cash provided by operating activities'])
        capex = find_numeric(['capital expenditure', 'capital expenditures', 'capex', 'purchase of property, plant and equipment'])
        if ocf is not None and capex is not None:
            last_fcf = ocf - abs(capex)  # capex often negative in table
        else:
            raise ValueError(f"No Free Cash Flow (or OCF/CapEx fallback) found for {ticker} on stockanalysis; page layout may have changed.")

    # Yahoo growth estimate (best-effort)
    url = f"https://finance.yahoo.com/quote/{ticker}/analysis?p={ticker}"
    r = requests.get(url, headers=headers, timeout=15)
    ge = []
    if r.status_code == 200:
        parser = html.fromstring(r.content)
        rows_y = parser.xpath('//table//tbody//tr') or []
        for row in rows_y:
            labels = [t.strip() for t in row.xpath("td/span/text()") if t.strip()]
            vals = [t.strip() for t in row.xpath("td/text()") if t.strip()]
            if labels and 'Next 5 Years' in labels[0]:
                try:
                    ge = float(vals[0].replace('%', '').strip())
                except Exception:
                    ge = []
                break

    # shares
    url = f"https://stockanalysis.com/stocks/{ticker}/"
    r = requests.get(url, headers=headers, verify=False, timeout=15)
    if r.status_code != 200:
        raise ConnectionError(f"Failed to fetch {url} (status {r.status_code})")
    parser = html.fromstring(r.content)
    shares_rows = parser.xpath('//table[@data-test="overview-info"]//tbody//tr[td/text()[contains(., "Shares Out")]]')
    shares_text = first_or("Shares Out row not found", shares_rows).xpath('td/text()')[2]
    factor = 1
    if 'B' in shares_text:
        factor = 1_000_000_000
    elif 'M' in shares_text:
        factor = 1_000_000
    shares = float(shares_text.replace('B','').replace('M','').replace(',','')) * factor

    # financials page for eps / market price
    url = f"https://stockanalysis.com/stocks/{ticker}/financials/"
    r = requests.get(url, headers=headers, verify=False, timeout=15)
    parser = html.fromstring(r.content)
    eps_vals = parser.xpath('//table[contains(@data-test,"financials")]//tr[td/span/text()[contains(., "EPS (Diluted)")]]') or []
    eps_text = first_or("EPS (Diluted) row not found", eps_vals).xpath('.//td/text()')[1]
    eps = float(eps_text.replace(",", ""))

    price_nodes = parser.xpath('//div[@class="price-ext"]/text()') or []
    mp = float(first_or("Market price not found", price_nodes).replace('$','').replace(',',''))

    return {'fcf': last_fcf, 'ge': ge, 'yr': 5, 'dr': 10, 'pr': 2.5, 'shares': shares, 'eps': eps, 'mp': mp}
# ...existing code...

def dcf(data):
    forecast = [data['fcf']]
    
    if data.get('ge', []) == []:
        raise ValueError("No growth rate available from Yahoo Finance (data['ge'] is empty) - provide growth_estimate to main() or fix parsing.")
    # ...existing dcf body...
# ...existing code...

    for i in range(1, data['yr']):
        forecast.append(round(forecast[-1] + (data['ge'] / 100) * forecast[-1], 2))

    forecast.append(round(forecast[-1] * (1 + (data['pr'] / 100)) / (data['dr'] / 100 - data['pr'] / 100), 2)) #terminal value
    discount_factors = [1 / (1 + (data['dr'] / 100))**(i + 1) for i in range(len(forecast) - 1)]

    pvs = [round(f * d, 2) for f, d in zip(forecast[:-1], discount_factors)]
    pvs.append(round(discount_factors[-1] * forecast[-1], 2)) # discounted terminal value
    
    print("Forecasted cash flows: {}".format(", ".join(map(str, forecast))))
    print("PV of cash flows: {}".format(", ".join(map(str, pvs))))

    dcf = sum(pvs)
    print("Fair value: {}\n".format(dcf / data['shares']))

'''
def dcf(data):
    forecast = [data['fcf']]

    if data['ge'] == []:
        raise ValueError("No growth rate available from Yahoo Finance")
'''

    

def reverse_dcf(data):
    pass

def graham(data):
    if data['eps'] > 0:
        expected_value = data['eps'] * (8.5 + 2 * (data['ge']))
        ge_priced_in = (data['mp'] / data['eps'] - 8.5) / 2

        print("Expected value based on growth rate: {}".format(expected_value))
        print("Growth rate priced in for next 7-10 years: {}\n".format(ge_priced_in))
    else:
        print("Not applicable since EPS is negative.")

'''
if __name__ == "__main__":
    argparser = argparse.ArgumentParser()
    argparser.add_argument('ticker', help='Ticker to analyse. Example: GOOG')
    argparser.add_argument('--discount_rate', help='Discount rate in %. Default: 10', default=10)
    argparser.add_argument('--growth_estimate', help='Estimated yoy growth rate. Default: Fetched from Yahoo Finance')
    argparser.add_argument('--terminal_rate', help='Terminal growth rate. Default: 2.5')
    argparser.add_argument('--period', help='Time period in years. Default: 5')
    args = argparser.parse_args()
    
    ticker = args.ticker

    print("Fetching data for %s...\n" % (ticker))
    data = parse(ticker)
    print("=" * 80)
    print("DCF model (basic)")
    print("=" * 80 + "\n")

    if args.period is not None:
        data['yr'] = int(args.period)
    if args.growth_estimate is not None:
        data['ge'] = float(args.growth_estimate)
    if args.discount_rate is not None:
        data['dr'] = float(args.discount_rate)
    if args.terminal_rate is not None:
        data['pr'] = float(args.terminal_rate)

    print("Market price: {}".format(data['mp']))
    print("EPS: {}".format(data['eps']))
    print("Growth estimate: {}".format(data['ge']))
    print("Term: {} years".format(data['yr']))
    print("Discount Rate: {}%".format(data['dr']))
    print("Perpetual Rate: {}%\n".format(data['pr']))

    dcf(data)

    print("=" * 80)
    print("Graham style valuation basic (Page 295, The Intelligent Investor)")
    print("=" * 80 + "\n")

    graham(data)
    
'''

'\nif __name__ == "__main__":\n    argparser = argparse.ArgumentParser()\n    argparser.add_argument(\'ticker\', help=\'Ticker to analyse. Example: GOOG\')\n    argparser.add_argument(\'--discount_rate\', help=\'Discount rate in %. Default: 10\', default=10)\n    argparser.add_argument(\'--growth_estimate\', help=\'Estimated yoy growth rate. Default: Fetched from Yahoo Finance\')\n    argparser.add_argument(\'--terminal_rate\', help=\'Terminal growth rate. Default: 2.5\')\n    argparser.add_argument(\'--period\', help=\'Time period in years. Default: 5\')\n    args = argparser.parse_args()\n    \n    ticker = args.ticker\n\n    print("Fetching data for %s...\n" % (ticker))\n    data = parse(ticker)\n    print("=" * 80)\n    print("DCF model (basic)")\n    print("=" * 80 + "\n")\n\n    if args.period is not None:\n        data[\'yr\'] = int(args.period)\n    if args.growth_estimate is not None:\n        data[\'ge\'] = float(args.growth_estimate)\n    if args.discount_rate is not None:\n

In [13]:
# ...existing code...
def main(ticker, discount_rate=10, growth_estimate=None, terminal_rate=2.5, period=5):
    """
    Notebook-friendly entry point. Call main('AAPL', ...) from a cell.
    """
    try:
        data = parse(ticker)
    except Exception as e:
        print(f"Error fetching/parsing data for {ticker}: {e}")
        return

    # override defaults if provided
    if period is not None:
        data['yr'] = int(period)
    if growth_estimate is not None:
        data['ge'] = float(growth_estimate)
    if discount_rate is not None:
        data['dr'] = float(discount_rate)
    if terminal_rate is not None:
        data['pr'] = float(terminal_rate)

    print(f"Fetching data for {ticker}...\n")
    print("=" * 80)
    print("DCF model (basic)")
    print("=" * 80 + "\n")

    print("Market price: {}".format(data['mp']))
    print("EPS: {}".format(data['eps']))
    print("Growth estimate: {}".format(data['ge']))
    print("Term: {} years".format(data['yr']))
    print("Discount Rate: {}%".format(data['dr']))
    print("Perpetual Rate: {}%\n".format(data['pr']))

    dcf(data)

    print("=" * 80)
    print("Graham style valuation basic (Page 295, The Intelligent Investor)")
    print("=" * 80 + "\n")

    graham(data)

# Example usage in a notebook cell:
# main('AAPL')                        # use fetched defaults
# main('AAPL', discount_rate=9)       # override discount rate
# main('AAPL', growth_estimate=12)    # override growth estimate
# ...existing code...

In [14]:
main('AAPL')

Error fetching/parsing data for AAPL: No Free Cash Flow (or OCF/CapEx fallback) found for AAPL on stockanalysis; page layout may have changed.
